# Identify 3D templates for RNA targets
This notebook prepares a mock submission as an illustration of how to derive 3D templates for targets in the Stanford 3D RNA folding competition. See also https://www.kaggle.com/datasets/rhijudas/rna-3d-folding-templates/data for precomputed outputs useful for training models for RNA 3D folding.

## Important options

In [ ]:
# if False, don't check on temporal_cutoff -- during training on known structures, will get lots of leakage. 
# If going for Early Sharing Prize, set to True.
CHECK_TEMPORAL_CUTOFF = True

# Number of templates to use. 
# Here set to 5 to prepare a mock submission
# Should make larger (e.g., 40) if using templates for modeling. 
MAX_TEMPLATES = 5

# Better to use nan when preparing files for templates, to allow easy recognition of which coordinates are missing.
# But for this example, using 0.0 to avoid errors in scoring the final submission.csv
NULL_VALUE = 0.0  


## Download MMseqs2

In [ ]:
#!wget https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz
#!tar xvfz /kaggle/working/mmseqs-linux-avx2.tar.gz
!rsync -avL /kaggle/input/mmseqs2/mmseqs /kaggle/working/
!chmod 755 /kaggle/working/mmseqs/bin/mmseqs

## Create DB based on FASTA of all PDB nucleic acid sequences, which is part of PDB_RNA dataset

In [ ]:
!/kaggle/working/mmseqs/bin/mmseqs createdb /kaggle/input/stanford-rna-3d-folding-2/PDB_RNA/pdb_seqres_NA.fasta pdb_seqres_NA --dbtype 2 # > MMseqs_createDB.log

## Find templates for targets in test_sequences.csv by aligning to PDB with MMseqs2

### Need to  convert test_sequences.csv file to FASTA

In [ ]:
import csv
input_file='/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv'
output_file='test_sequences.fasta'
with open(input_file, 'r', newline='') as csv_file, open(output_file, 'w') as fasta_file:
    csv_reader = csv.reader(csv_file, quotechar='"', delimiter=',', quoting=csv.QUOTE_ALL, skipinitialspace=True)
    next(csv_reader)  # Skip the header row
    for row in csv_reader:
        if len(row) >= 2:
            fasta_file.write(f">{row[0]}\n{row[1]}\n")

In [ ]:
!/kaggle/working/mmseqs/bin/mmseqs easy-search /kaggle/working/test_sequences.fasta /kaggle/working/pdb_seqres_NA testResult.txt tmp --search-type 3 --format-output "query,target,evalue,qstart,qend,tstart,tend,qaln,taln" 

## Assemble file of template coordinates by going through .cif files found for each target by MMseqs2

In [ ]:
!pip3 install /kaggle/input/biopython/biopython-1.85-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --no-deps

In [ ]:
from Bio import SeqIO,PDB,BiopythonWarning
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio.Seq import Seq
from Bio.PDB import MMCIFParser
import numpy as np
import pandas as pd
import os
import gzip
import sys
import warnings
import argparse
from datetime import datetime

# Suppress warnings
warnings.simplefilter('ignore', BiopythonWarning)

### Basic options and file locations

In [ ]:
sequences_file = '/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv'
mmseqs_results_file = '/kaggle/working/testResult.txt' # created by MMseqs command lines
outfile = 'submission.csv'
cif_dir = '/kaggle/input/stanford-rna-3d-folding-2/PDB_RNA'

# variables not in use here:
id_map_file = ''
start_idx = 0
end_idx = 0

### Helper functions

In [ ]:
def clean_res_name( res_name ):
    if res_name in ['A', 'C', 'G', 'U']:
        return res_name
    else: # can be modified residue with 3-letter name.
        return 'X'

def extract_title_release_date( cif_path ):

    if cif_path.endswith('.gz'):
        with gzip.open(cif_path, 'rt') as cif_file:
            mmcif_dict = MMCIF2Dict(cif_file)
    else:
        mmcif_dict = MMCIF2Dict(cif_path)

    possible_title_fields = [
        '_struct.title',
        '_entry.title',
        '_struct_keywords.pdbx_keywords'
    ]

    pdb_title = None
    for field in possible_title_fields:
        if field in mmcif_dict:
            pdb_title = mmcif_dict[field]
            if isinstance(pdb_title, list):
                pdb_title = ' '.join(pdb_title)
            break

    possible_date_fields = [
        '_pdbx_database_status.initial_release_date',
        '_pdbx_database_status.recvd_initial_deposition_date',
        '_database_PDB_rev.date'
    ]

    release_date = None
    for field in possible_date_fields:
        if field in mmcif_dict:
            release_date = mmcif_dict[field]
            if isinstance(release_date, list):
                release_date = release_date[0]  # Take the first date if it's a list
            break

    return pdb_title, release_date


def extract_rna_sequence(cif_path,chain_id):

    if cif_path.endswith('.gz'):
        with gzip.open(cif_path, 'rt') as cif_file:
            mmcif_dict = MMCIF2Dict(cif_file)
    else:
        mmcif_dict = MMCIF2Dict(cif_path)

    pdb_sequence = None
    pdb_chain_id = None
    chain_seq_nums = None

    # Extract _pdbx_poly_seq_scheme information
    strand_id  = mmcif_dict.get('_pdbx_poly_seq_scheme.pdb_strand_id',[])
    mon_id     = mmcif_dict.get('_pdbx_poly_seq_scheme.mon_id',[])
    pdb_mon_id = mmcif_dict.get('_pdbx_poly_seq_scheme.pdb_mon_id',[])
    pdb_seq_num = mmcif_dict.get('_pdbx_poly_seq_scheme.pdb_seq_num',[])
    chain_ids = list(set(strand_id))
    seq_chains = []

    full_sequence = ''
    pdb_chain_sequence = ''
    pdb_chain_seq_nums = []
    for (strand,mon,pdb_mon,pdb_num) in zip(strand_id,mon_id,pdb_mon_id,pdb_seq_num):
        if strand==chain_id:
            full_sequence += clean_res_name( mon )
            pdb_chain_sequence += clean_res_name( pdb_mon )
            pdb_chain_seq_nums.append( pdb_num)

    #print(full_sequence)
    #print(pdb_chain_sequence)
    #print(pdb_chain_seq_nums)

    return full_sequence,pdb_chain_sequence,pdb_chain_seq_nums

def get_c1prime_labels(cif_path, chain_id, alignment, chain_seq_nums):
    """
    Extract C1' coordinates for an RNA chain based on a reference sequence alignment.

    This function uses Biopython to parse a CIF file, finds the specified chain,
    and extracts C1' coordinates for RNA residues. It aligns these coordinates
    with a reference sequence, handling gaps and missing residues.

    Parameters:
    cif_path (str): Path to the CIF file.
    chain_id (str): Chain identifier in the CIF file.
    alignment (list): A list containing two elements:
                      alignment[0]: List of residues for the reference sequence (A,C,G,U,-)
                      alignment[1]: List of residues for the chain sequence (A,C,G,U,X,-)
    chain_seq_nums (list): numbers of residues in PDB

    Returns:
    list of tuples: Each tuple contains (resname, resid, x, y, z), where:
                    resname: Residue name (A, C, G, or U) from the reference sequence
                    resid: Residue ID (1, 2, 3, ...) based on position in reference sequence
                    x, y, z: C1' coordinates (nan or NULL_VALUE for missing residues/atoms)

    The length of the returned list is equal to the number of non-gap residues
    in the reference sequence.
    """
    # Parse the CIF file
    parser = MMCIFParser()
    if cif_path.endswith('.gz'):
        with gzip.open(cif_path, 'rt') as gz_file:
            structure = parser.get_structure('RNA', gz_file )
    else:
        structure = parser.get_structure('RNA', cif_path)

    # Get the specified chain
    chain = structure[0][chain_id]

    # getting residues out of chain is complex -- easier to get a list ahead of time.
    residues = {}
    for residue in chain: residues[ residue.id[1] ] = residue

    chain_seq = ''.join( [clean_res_name(residue.get_resname()) for residue in chain ] )
    #print(chain_seq)

    # Initialize the result list
    result = []

    # Counter for residue ID in reference sequence
    ref_resid = 0
    chain_idx = 0
    for ref_res, chain_res in zip(alignment[0], alignment[1]):
        if chain_res != '-': chain_idx += 1
        if ref_res != '-':
            ref_resid += 1
            if chain_res == '-': # or chain_res == 'X':
                # Missing residue in chain or unknown residue
                result.append((ref_res, ref_resid, NULL_VALUE, NULL_VALUE, NULL_VALUE, -1e18))
            else:
                # Find the corresponding residue in the chain
                try:
                    #chain_seq_num = int(chain_seq_nums[ref_resid-1])
                    chain_seq_num = int(chain_seq_nums[chain_idx-1])
                    residue = residues[chain_seq_num]
                    c1_prime = residue['C1\'']
                    coords = c1_prime.coord
                    if residue.get_resname() != chain_res:
                        print( 'WARNING!',ref_resid,chain_idx,chain_seq_num,residue.get_resname(),chain_res)
                    result.append((ref_res, ref_resid, coords[0], coords[1], coords[2], residue.id[1]))
                except KeyError:
                    # C1' atom not found
                    result.append((ref_res, ref_resid, NULL_VALUE, NULL_VALUE, NULL_VALUE, -1e18))
                except Exception as e:
                    # Any other error (e.g., residue not found)
                    result.append((ref_res, ref_resid, NULL_VALUE, NULL_VALUE, NULL_VALUE, -1e18))

    return result

def is_before_or_on(d1, d2):
    date1 = pd.to_datetime(d1)
    date2 = pd.to_datetime(d2)
    return date1 <= date2

def read_id_map(id_map_file):
    if len(id_map_file)==0: return None
    id_map = {}
    try:
        with open(id_map_file, newline='') as f:
            reader = csv.DictReader(f)
            if 'orig' not in reader.fieldnames or 'new' not in reader.fieldnames:
                print("Warning: ID map file does not contain the fields 'orig' and 'new'. Using original IDs instead.")
                return id_map
            for row in reader:
                id_map[row['orig']] = row['new']
    except FileNotFoundError:
        print(f"Warning: ID map file {id_map_file} not found. Using original IDs instead.", file=sys.stderr)
    except Exception as exc:
        print(f"Error reading {id_map_file}: {exc}", file=sys.stderr)
    return id_map

def read_release_dates( release_data_file ):
    release_dates = {}
    # must have format Entry ID, Release Date
    with open(release_data_file, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            release_dates[row['Entry ID']] = row['Release Date']

    return release_dates

### Setup

In [ ]:
# Prepare to collect output data
output_labels = []

# Read the FASTA file
df = pd.read_csv( sequences_file )
targets = df['target_id'].to_list()
sequences = df['sequence'].to_list()
temporal_cutoffs = df['temporal_cutoff'].to_list()

aln_lines = []
for line in open( mmseqs_results_file ).readlines():
    # query,template,eval,qstart,qend,tstart,tend,qaln,taln
    aln_lines.append( line.strip().split() )

id_map = read_id_map( id_map_file )

release_dates = read_release_dates( cif_dir + '/pdb_release_dates_NA.csv' )

if start_idx == 0 and end_idx == 0: # do all targets by default
    start_idx = 1
    end_idx = len(targets)

In [ ]:
num_targets = 0
count = 0
for target,sequence,temporal_cutoff in zip(targets,sequences,temporal_cutoffs):
    count += 1
    if (count < start_idx) or (count > end_idx): continue

    # look for alignments and fill out C1' templates
    templates = []
    for aln_line in aln_lines:
        if len(aln_line)!=9: continue # some kind of overflow in some alignments?

        query,template,eval,qstart,qend,tstart,tend,qaln,taln = aln_line

        if query != target: continue

        if int(qend)<int(qstart): continue # aligned to reverse complement!

        pdb_id,chain_id = template.split('_')

        # need to do alignment
        cif_path = os.path.join(cif_dir, f'{pdb_id.lower()}.cif')
        if not os.path.isfile( cif_path ): continue # occasional alignment to DNA, ignore!

        release_date = release_dates[pdb_id.upper()] # pulled from PDB server

        if CHECK_TEMPORAL_CUTOFF and is_before_or_on(temporal_cutoff,release_date): continue

        # these release dates in the CIF files can be buggy!
        title,release_date_unreliable = extract_title_release_date( cif_path )

        print('\n',target,temporal_cutoff,"   ",template)
        if title: print(f"PDB Title: {title}")
        if release_date: print(f"PDB Release Date: {release_date}")

        # sometimes there is a mismatch between PDB's fasta files and what's actually stored in coordinates,
        # so best to get the actual residue numbers for the chain
        chain_full_sequence,chain_sequence,chain_seq_nums = extract_rna_sequence(cif_path,chain_id)

        # get 3d data
        alignment = []
        qstart=int(qstart)
        qend=int(qend)
        tstart=int(tstart)
        tend=int(tend)
        alignment.append( sequence[:(qstart-1)] + '-'*(tstart-1) + qaln + sequence[qend:]  )
        alignment.append( '-'*(qstart-1)        + 'X'*(tstart-1) + taln + '-'*(len(sequence)-qend) )
        print( alignment[0] )
        print( alignment[1] )
        c1prime_data = get_c1prime_labels( cif_path, chain_id, alignment, chain_seq_nums )

        # mismatch in FASTA sequence and the polyx info in the CIF file
        if len(c1prime_data) != len(sequence):
            print( 'WARNING! len(c1prime_data) != len(sequence)', 'len c1prime_data', len(c1prime_data), 'len sequence', len(sequence), 'qstart',qstart,'len qaln',len(qaln),'qend',qend)
            continue

        templates.append( c1prime_data )

        if len(templates) >= MAX_TEMPLATES: break

    print( "Found", len(templates), "templates for", target,'\n' )

    mapped_target = target
    if not id_map is None: mapped_target = id_map[target]

    for i in range(len(sequence)):
        output_label = {
            "ID": f'{mapped_target}_{i+1}',
            "resname": sequence[i],
            "resid": i+1,
        }

        # output templates
        for n in range(len(templates)):
            res,resid,x,y,z,pdb_seqnum = templates[n][i]
            assert( resid == i+1 )
            output_label[ f"x_{n+1}" ] = x
            output_label[ f"y_{n+1}" ] = y
            output_label[ f"z_{n+1}" ] = z

        # pad with blank models
        for n in range(len(templates),MAX_TEMPLATES):
            output_label[ f"x_{n+1}" ] = NULL_VALUE
            output_label[ f"y_{n+1}" ] = NULL_VALUE
            output_label[ f"z_{n+1}" ] = NULL_VALUE
        output_labels.append( output_label )

    num_targets += 1
    # if num_targets > 1: break # for debug!


print(f'Completed {num_targets} targets\n')


In [ ]:
# Create a DataFrame and write to CSV
def output_csv( output_data, outfile ):
    df = pd.DataFrame(output_data)
    df.to_csv(outfile, index=False)
    print(f"Output written to {outfile}")

output_csv( output_labels, outfile )

# Let's run evaluation with the competition's metric to get scores 

This code block reads in the metric **TM-Score PermuteChains**, which can be installed as input data for the notebook from the link https://www.kaggle.com/code/rhijudas/tm-score-permutechains/data. 

We also need the USalign code installed as input data, available at: https://www.kaggle.com/datasets/metric/usalign

In [ ]:
!ls /kaggle/usr/lib/
import runpy
module_globals = runpy.run_path("/kaggle/usr/lib/tm-score-permutechains/metric.py")
score = module_globals['score']

In [ ]:
import pandas as pd
sol = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/validation_labels.csv')
sub = pd.read_csv(outfile)
print('Length of solution:',len(sol),'Length of submission',len(sub))

The score() function will go over entire dataframe -- all the targets. But let's get scores target by target so we can get feedback on which ones are 'solved' vs. not. 

In [ ]:
sol['target_id'] = sol['ID'].apply(lambda x: '_'.join(str(x).split('_')[:-1]))
sub['target_id'] = sub['ID'].apply(lambda x: '_'.join(str(x).split('_')[:-1]))

if len(sol)==len(sub): # This tests if we're looking at public val
    results = []
    for target_id, group_native in sol.groupby('target_id'):
        # if len(group_native)>200: continue # quick test
        group_predicted = sub[sub['target_id'] == target_id]
        result = score(group_native,group_predicted,'ID')
        print(target_id,result)
        results.append( result )
    print( 'Mean score:',  
          float(sum(results) / len(results)) if len(results)>0 else 0.0, 
          f'(n={len(results)})' )